**Install PyGithub to access Github Rest API**

In [ ]:
!pip install PyGithub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.7/432.7 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 32.0 MB/s eta 0:00:00


**Import necessary python modules**

In [ ]:
from github import Github, Auth
import csv
import logging
from datetime import date, timedelta, datetime
from google.colab import files
import json
import re
import markdown
from bs4 import BeautifulSoup

logging.getLogger("github").setLevel(logging.WARNING)

**Authorization**

In [ ]:
# GitHub Personal Access Token
GITHUB_TOKEN = ""

# Use new Auth method
auth = Auth.Token(GITHUB_TOKEN)

# Initialize GitHub client with the new auth object
g = Github(auth=auth)

**Apply Search Query with Filters and Slicing Period**

In [ ]:
def generate_semi_year_slices(start_date, end_date):
    """
    Generate 6-month (semiannual) date ranges.
    """
    start = date.fromisoformat(start_date)
    end = date.fromisoformat(end_date)

    slices = []
    current = start

    while current <= end:
        if current.month <= 6:
            slice_start = date(current.year, 1, 1)
            slice_end = date(current.year, 6, 30)
            next_start = date(current.year, 7, 1)
        else:
            slice_start = date(current.year, 7, 1)
            slice_end = date(current.year, 12, 31)
            next_start = date(current.year + 1, 1, 1)

        if slice_start < start:
            slice_start = start
        if slice_end > end:
            slice_end = end

        slices.append((slice_start.isoformat(), slice_end.isoformat()))
        current = next_start

    return slices

In [ ]:
def generate_quarter_slices(start_date, end_date):
    """
    Generate 3-month (quarter) date ranges.
    """
    start = date.fromisoformat(start_date)
    end = date.fromisoformat(end_date)

    slices = []
    current = start

    while current <= end:
        quarter = (current.month - 1) // 3
        q_start_month = quarter * 3 + 1
        q_end_month = q_start_month + 2

        q_start = date(current.year, q_start_month, 1)

        # compute end of quarter
        if q_end_month == 12:
            q_end = date(current.year, 12, 31)
            next_start = date(current.year + 1, 1, 1)
        else:
            next_start = date(current.year, q_end_month + 1, 1)
            q_end = next_start - timedelta(days=1)

        # clip to bounds
        if q_start < start:
            q_start = start
        if q_end > end:
            q_end = end

        slices.append((q_start.isoformat(), q_end.isoformat()))
        current = next_start

    return slices

In [ ]:
def generate_monthly_slices(start_date, end_date):
    """
    Generate monthly date ranges.
    """
    start = date.fromisoformat(start_date)
    end = date.fromisoformat(end_date)

    slices = []
    current = start.replace(day=1)

    while current <= end:
        # first day of next month
        if current.month == 12:
            next_month = date(current.year + 1, 1, 1)
        else:
            next_month = date(current.year, current.month + 1, 1)

        month_start = current
        month_end = next_month - timedelta(days=1)

        # clip to bounds
        if month_start < start:
            month_start = start
        if month_end > end:
            month_end = end

        slices.append((month_start.isoformat(), month_end.isoformat()))
        current = next_month

    return slices

In [ ]:
def generate_partialmonth_slices(start_date, end_date):
    """
    Generate 15-day date ranges.
    """
    start = date.fromisoformat(start_date)
    end = date.fromisoformat(end_date)

    slices = []
    current = start

    while current <= end:
        slice_start = current
        slice_end = min(current + timedelta(days=14), end)

        slices.append((
            slice_start.isoformat(),
            slice_end.isoformat()
        ))

        current = slice_end + timedelta(days=1)

    return slices

In [ ]:
def get_readme_snippet(repo):
    """
    Extracts the first heading and first paragraph from the README
    and combines them into a single string.
    """
    heading = ""
    paragraph = ""

    try:
        readme = repo.get_readme()
        md_content = readme.decoded_content.decode("utf-8", errors="ignore")
    except Exception:
        return ""  # No README

    # Convert markdown to HTML
    html = markdown.markdown(md_content)
    soup = BeautifulSoup(html, "html.parser")

    # First heading
    for h_tag in ["h1", "h2", "h3", "h4", "h5", "h6"]:
        h = soup.find(h_tag)
        if h and h.get_text(strip=True):
            heading = h.get_text(strip=True)
            break

    # First paragraph with a sentence
    for p in soup.find_all("p"):
        text = p.get_text(strip=True)
        if text and re.search(r"[.!?。！？]", text):
            paragraph = text
            break

    # Combine heading + paragraph
    combined_text = heading
    if paragraph:
        if combined_text:
            combined_text += " "
        combined_text += paragraph

    return combined_text

In [ ]:
def compute_language_percentages(repo, other_threshold=1.0):
    """
    Compute GitHub-like language percentages.
    Languages below `other_threshold` (%) are grouped into 'Other'.
    """
    try:
        langs = repo.get_languages()
    except Exception:
        return "{}"

    total_bytes = sum(langs.values())
    if total_bytes == 0:
        return "{}"

    percentages = {
        lang: (bytes_ / total_bytes) * 100
        for lang, bytes_ in langs.items()
    }

    major = {}
    other_sum = 0.0

    for lang, pct in percentages.items():
        if pct >= other_threshold:
            major[lang] = round(pct, 1)
        else:
            other_sum += pct

    if other_sum > 0:
        major["Other"] = round(other_sum, 1)

    return json.dumps(major)

**Mine Github Repositories**

In [ ]:
def mine_repositories(g, llm_keywords, agent_keywords, base_filters, start_date, end_date):
    """
    Mine GitHub repositories with adaptive time slicing.
    start_date, end_date: YYYY-MM-DD

    Returns:
        dict: {repo.full_name: repo}
    """
    all_repos = {}

    start = date.fromisoformat(start_date)
    end = date.fromisoformat(end_date)

    for llm in llm_keywords:
        for agent in agent_keywords:

            for year in range(start.year, end.year + 1):

                # Adjust year boundaries
                if year == start.year:
                    y_start = start.isoformat()
                else:
                    y_start = f"{year}-01-01"

                if year == end.year:
                    y_end = end.isoformat()
                else:
                    y_end = f"{year}-12-31"

                year_query = f'{llm} {agent} {base_filters} created:{y_start}..{y_end}'
                year_results = g.search_repositories(query=year_query)
                print("\nRunning query:", year_query)
                print(f"Year slice {year}: {year_results.totalCount}")

                if year_results.totalCount < 1000:
                    for repo in year_results:
                        all_repos[repo.full_name] = repo
                    continue

                # Semi-year slicing
                for s_start, s_end in generate_semi_year_slices(y_start, y_end):
                    semi_query = f'{llm} {agent} {base_filters} created:{s_start}..{s_end}'
                    semi_results = g.search_repositories(query=semi_query)
                    print(f"Semi-yearly slice {s_start}..{s_end}: {semi_results.totalCount}")

                    if semi_results.totalCount < 1000:
                        for repo in semi_results:
                            all_repos[repo.full_name] = repo
                        continue

                    # Quarter slicing
                    for q_start, q_end in generate_quarter_slices(s_start, s_end):
                        quarter_query = f'{llm} {agent} {base_filters} created:{q_start}..{q_end}'
                        quarter_results = g.search_repositories(query=quarter_query)
                        print(f"Quarterly slice {q_start}..{q_end}: {quarter_results.totalCount}")

                        if quarter_results.totalCount < 1000:
                            for repo in quarter_results:
                                all_repos[repo.full_name] = repo
                            continue

                        # Monthly slicing
                        for m_start, m_end in generate_monthly_slices(q_start, q_end):
                            month_query = f'{llm} {agent} {base_filters} created:{m_start}..{m_end}'
                            month_results = g.search_repositories(query=month_query)
                            print(f"Monthly slice {m_start}..{m_end}: {month_results.totalCount}")

                            if month_results.totalCount < 1000:
                                for repo in month_results:
                                    all_repos[repo.full_name] = repo
                                continue

                            # 15-day slicing
                            for f_start, f_end in generate_partialmonth_slices(m_start, m_end):
                                partialmonth_query = f'{llm} {agent} {base_filters} created:{f_start}..{f_end}'
                                partialmonth_results = g.search_repositories(query=partialmonth_query)
                                print(f"15-day slice {f_start}..{f_end}: {partialmonth_results.totalCount}")

                                for repo in partialmonth_results:
                                    all_repos[repo.full_name] = repo

    return all_repos

In [ ]:
def clean_readme(text):
    if not text:
        return ""

    # Remove HTML tags
    soup = BeautifulSoup(text, "html.parser")
    clean_text = soup.get_text(separator="\n")

    # Remove linked images: [![alt](img)](link)
    clean_text = re.sub(r"\[\s*!\[.*?\]\(.*?\)\s*\]\(.*?\)", "", clean_text)

    # Remove standalone images: ![alt](img)
    clean_text = re.sub(r"!\[.*?\]\(.*?\)", "", clean_text)

    # Remove empty markdown links: [](url)
    clean_text = re.sub(r"\[\s*\]\(.*?\)", "", clean_text)

    # Convert normal links to text: [text](url) → text
    clean_text = re.sub(r"\[([^\]]+)\]\([^)]+\)", r"\1", clean_text)

    # Remove markdown bullets with no content
    clean_text = re.sub(r"^\s*[-•*]\s*$", "", clean_text, flags=re.MULTILINE)

    # Remove code blocks
    clean_text = re.sub(r"`{3}.*?`{3}", "", clean_text, flags=re.S)
    clean_text = re.sub(r"`[^`]+`", "", clean_text)

    # Normalize whitespace
    clean_text = re.sub(r"\n{3,}", "\n\n", clean_text)
    clean_text = re.sub(r"[ \t]+", " ", clean_text)

    return clean_text.strip()


**Export the results to CSVs file**

In [ ]:
import base64

In [ ]:
def write_repos_to_csv(all_repos, output_file):
    """
    Write GitHub repository data to a CSV file with extended metadata.
    """
    with open(output_file, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)

        # Header
        writer.writerow([
            "full_name",
            "description",
            "stars",
            "forks",
            "language",
            "url",
            "updated_year",
            "last_updated",
            "created_year",
            "created_at",
            "watchers_count",
            "size",
            "license",
            "languages",
            "primary_language",
            "topics",
            "readme_content"
        ])

        # Rows
        for name, repo in all_repos.items():
            updated_year = repo.pushed_at.year if repo.pushed_at else ""
            last_updated = repo.pushed_at.isoformat() if repo.pushed_at else ""
            created_year = repo.created_at.year if repo.created_at else ""
            created_at = repo.created_at.isoformat() if repo.created_at else ""

            watchers_count = repo.watchers_count
            size = repo.size  # in KB
            license_name = repo.license.name if repo.license else ""
            primary_language = repo.language or ""

            # Topics
            try:
                topics = repo.get_topics()
            except Exception:
                topics = ""

            # All languages used
            try:
                languages = repo.get_languages()
            except Exception:
                languages = ""

            try:
                description = repo.description or ""
            except Exception:
                description = ""

            # README (FULL CONTENT)
            try:
                readme = repo.get_readme()
                readme_content = base64.b64decode(readme.content).decode("utf-8", errors="ignore")
                #readme_content = clean_readme(readme_content)
            except Exception:
                readme_content = ""


            writer.writerow([
                repo.full_name,
                description,
                repo.stargazers_count,
                repo.forks_count,
                primary_language,
                repo.html_url,
                updated_year,
                last_updated,
                created_year,
                created_at,
                watchers_count,
                size,
                license_name,
                languages,
                primary_language,
                topics,
                readme_content
            ])

    print(f"CSV file created: {output_file}")

In [ ]:
llm_keywords = ["LLM", '"large language model"']
agent_keywords = ["agent", "autonomous", '"multi-agent"', '"task automation"']

base_filters = "language:Python fork:false archived:false"


**Results**

*   CSV file 1: Agent Repositories in Python language
*   CSV file 2: Agent Repositories in other languages not Python



In [ ]:
today_str = datetime.today().strftime("%Y-%m-%d")
print(today_str)

2026-01-12


In [ ]:
all_repos = mine_repositories(
    g=g,
    llm_keywords=llm_keywords,
    agent_keywords=agent_keywords,
    base_filters=base_filters,
    start_date="2021-06-01",
    end_date="2025-12-25"
)


Running query: LLM agent language:Python fork:false archived:false created:2021-06-01..2021-12-31
Year slice 2021: 2

Running query: LLM agent language:Python fork:false archived:false created:2022-01-01..2022-12-31
Year slice 2022: 12

Running query: LLM agent language:Python fork:false archived:false created:2023-01-01..2023-12-31
Year slice 2023: 516

Running query: LLM agent language:Python fork:false archived:false created:2024-01-01..2024-12-31
Year slice 2024: 1000
Semi-yearly slice 2024-01-01..2024-06-30: 774
Semi-yearly slice 2024-07-01..2024-12-31: 1000
Quarterly slice 2024-07-01..2024-09-30: 528
Quarterly slice 2024-10-01..2024-12-31: 771

Running query: LLM agent language:Python fork:false archived:false created:2025-01-01..2025-12-25
Year slice 2025: 1000
Semi-yearly slice 2025-01-01..2025-06-30: 1000
Quarterly slice 2025-01-01..2025-03-31: 1000
Monthly slice 2025-01-01..2025-01-31: 369
Monthly slice 2025-02-01..2025-02-28: 616
Monthly slice 2025-03-01..2025-03-31: 536
Qu

In [ ]:
output_file="csv/github_agent_repos_python_final.csv"
write_repos_to_csv(all_repos, output_file)

KeyboardInterrupt: 

In [ ]:
files.download(output_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>